# 🌡️ Delhi Deep Dive — Module 03: Heatwave Risk
## Extreme heat scoring for 11 Delhi districts (2010–2023)

In [ ]:
import os, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)
START, END = '2010-01-01', '2023-12-31'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

def fetch_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f'⏳ {wait}s…', end=' ', flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError('Max retries')

print('Heatwave module setup complete.')

## Fetch Temperature Data

In [ ]:
dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    params = {
        'latitude': coords['lat'], 'longitude': coords['lon'],
        'start_date': START, 'end_date': END,
        'daily': ['temperature_2m_max', 'apparent_temperature_max'],
        'timezone': 'Asia/Kolkata'
    }
    data = fetch_with_retry('https://archive-api.open-meteo.com/v1/archive', params)['daily']
    df = pd.DataFrame({
        'date': pd.to_datetime(data['time']),
        'temp_max_c': pd.to_numeric(data['temperature_2m_max'], errors='coerce').fillna(0),
        'apparent_temp_max_c': pd.to_numeric(data['apparent_temperature_max'], errors='coerce').fillna(0),
    })
    df['district'] = district
    df['lat'] = coords['lat']
    df['lon'] = coords['lon']
    dfs.append(df)
    print('✓')
    time.sleep(1.5)

daily = pd.concat(dfs, ignore_index=True)
daily['year'] = daily['date'].dt.year
print(f'\nTotal: {len(daily):,} rows | Max temp overall: {daily["temp_max_c"].max():.1f}°C')

## Feature Engineering + Train + Save

In [ ]:
def yearly_heat_features(g):
    g = g.sort_values('date')
    is_hot = g['temp_max_c'] > 40
    streaks = is_hot.groupby((~is_hot).cumsum()).sum()
    max_streak = int(streaks.max()) if is_hot.any() else 0
    return pd.Series({
        'days_above_40c':       int((g['temp_max_c'] > 40).sum()),
        'days_above_45c':       int((g['temp_max_c'] > 45).sum()),
        'max_temp_year':        float(g['temp_max_c'].max()),
        'heat_index_days':      int((g['apparent_temp_max_c'] > 45).sum()),
        'consecutive_hot_days': max_streak,
    })

yearly = daily.groupby(['district','year']).apply(yearly_heat_features).reset_index()
for d, c in DELHI_DISTRICTS.items():
    yearly.loc[yearly['district']==d, 'lat'] = c['lat']
    yearly.loc[yearly['district']==d, 'lon'] = c['lon']

yearly['heatwave_label'] = (yearly['days_above_40c'] > 10).astype(int)

FEATURES = ['days_above_45c','max_temp_year','heat_index_days','consecutive_hot_days']
train = yearly[yearly['year']<=2021].dropna(subset=FEATURES+['heatwave_label'])
test  = yearly[yearly['year']>=2022].dropna(subset=FEATURES+['heatwave_label'])

model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(train[FEATURES].astype(float), train['heatwave_label'].astype(int))

y_pred = model.predict(test[FEATURES].astype(float))
acc = accuracy_score(test['heatwave_label'], y_pred)
print(f'Test accuracy: {acc:.3f}')

full = yearly.dropna(subset=FEATURES+['heatwave_label']).copy()
probs = model.predict_proba(full[FEATURES].astype(float))[:,1]
full['heatwave_prob']       = probs.round(4)
full['heatwave_risk_score'] = (probs * 100).round(2)
full['risk_category']       = full['heatwave_risk_score'].apply(score_to_category)

# Feature importance
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance')
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(feat_df['feature'], feat_df['importance'], color='tomato')
ax.set_title('Delhi Heatwave — Feature Importances')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/delhi_heatwave_feature_importance.png', dpi=150)
plt.show()

result = full[['district','year','lat','lon','heatwave_label','heatwave_prob','heatwave_risk_score','risk_category']]
result.to_csv(f'{OUTPUT_DIR}/delhi_heatwave_scores.csv', index=False)
with open(f'{OUTPUT_DIR}/delhi_heatwave_model.pkl','wb') as f:
    pickle.dump(model, f)
print(f'Saved delhi_heatwave_scores.csv ({len(result)} rows)')
print(result[result['year']==2023][['district','heatwave_risk_score','risk_category']].to_string())